# Week 12 - Statistical Analysis

**Paññāsāstra University of Cambodia**
**Course:** Data Analysis and Visualization (3 credits)
**Chapter 12 - Statistical Analysis**

---

## Session topics (from the course outline)

- Descriptive statistics
- Probability distributions
- Hypothesis testing basics

## Course learning outcomes addressed

| CLO | Statement |
|---|---|
| **CLO 5** | Perform exploratory data analysis (EDA) to uncover insights |
| **CLO 8** | Evaluate the effectiveness of different visualization methods for communicating insights |
| **CLO 9** | Design interactive dashboards or visual reports to present data findings |

## Reading

| Topic | Source |
|---|---|
| Modelling libraries: statsmodels | McKinney §12.3 |
| Statistics with Python | Pfaffelhuber §9.1–9.5 |

## What this week adds

Every week so far has described a dataset: means, groups, charts, trends. Description answers
the question *what is in this sample*.

This week answers a different question: *what does this sample tell us about the population it
came from*. Those are not the same question, and the gap between them is where most published
mistakes live.

A concrete version. In Week 9 you compared exam scores across faculties and found a difference
of a few marks. Three things could be true:

1. The faculties genuinely differ, and your sample shows it.
2. The faculties are identical, and your sample differs by chance.
3. The faculties genuinely differ, but by so little that nobody should act on it.

A chart cannot separate these. Statistical analysis is the set of tools that can — and,
equally important, the set of habits that stop you from claiming more than the data supports.

---

## Setup

All imports for this notebook are defined in the single cell below. Run it first. If any
import fails, return to Notebook 00 and check that the kernel in the top-right corner points
inside your `venv`.

In [ ]:
# ---------------------------------------------------------------------------
# All imports for this notebook. Run this cell first.
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.power import TTestIndPower

# Reproducibility: every dataset in this notebook is generated from this seed,
# so your numbers will match the ones shown in the lecture exactly.
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# Consistent, colour-blind-safe plotting defaults for the whole course.
PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3", "#937860"]
plt.rcParams.update({
    "figure.figsize": (9, 4.5),
    "figure.dpi": 100,
    "axes.grid": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "grid.alpha": 0.3,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "font.size": 10,
})
sns.set_palette(PALETTE)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

print("Environment ready.")
print(f"  numpy   {np.__version__}")
print(f"  pandas  {pd.__version__}")
print(f"  seaborn {sns.__version__}")
print(f"  seed    {RANDOM_SEED}")

In [ ]:
# The two libraries added this week.
print(f"  scipy        {scipy.__version__}")
print(f"  statsmodels  {sm.__version__}")

---

# Part 1 — Theory and concepts

---

## 1.1 Population, sample, estimate

A **population** is every unit you would like to say something about: every student enrolled
at the university, every transaction a shop will ever take, every sensor reading a machine
will ever produce. You almost never have it.

A **sample** is the subset you actually observe. A **statistic** is any number computed from
the sample — a mean, a median, a correlation. A **parameter** is the corresponding number for
the population.

The convention is to write parameters with Greek letters and statistics with Latin ones:

| Quantity | Population parameter | Sample statistic |
|---|---|---|
| Mean | $\mu$ | $\bar{x}$ |
| Standard deviation | $\sigma$ | $s$ |
| Proportion | $\pi$ | $\hat{p}$ |
| Correlation | $\rho$ | $r$ |

The sample statistic is an **estimate** of the parameter. It is not the parameter. Draw a
second sample and you get a different number. The whole of this week follows from that one
observation.

---

## 1.2 Probability distributions, and the four things you do with them

A probability distribution describes how likely each value is. In `scipy.stats` every
distribution — `norm`, `t`, `chi2`, `binom`, `expon` and about a hundred others — exposes the
same small interface:

| Method | Name | Question it answers |
|---|---|---|
| `.pdf(x)` | density | How dense is the distribution at $x$? (continuous) |
| `.pmf(k)` | mass | What is $P(X = k)$? (discrete) |
| `.cdf(x)` | distribution function | What is $P(X \le x)$? |
| `.ppf(q)` | quantile, inverse CDF | Which $x$ has $P(X \le x) = q$? |
| `.sf(x)` | survival function | What is $P(X > x)$? |
| `.rvs(size)` | random variates | Draw a sample |

Learn this interface once and every distribution is available to you. Three families do almost
all the work this week:

- **Normal**, $N(\mu, \sigma^2)$. The distribution of a sum of many small independent effects.
  Sample means are approximately normal even when the data are not — this is the central limit
  theorem, and it is why the normal distribution appears in places the raw data never does.
- **Student's $t$**, with $\nu$ degrees of freedom. What you use instead of the normal when
  $\sigma$ is estimated from the data rather than known. It has heavier tails; as
  $\nu \rightarrow \infty$ it becomes the normal.
- **Chi-square**, $\chi^2_\nu$. The distribution of a sum of squared normal deviations, which
  is why it governs tests built from squared differences between counts.

---

## 1.3 The sampling distribution and the standard error

Imagine repeating your whole study many times, each time drawing a fresh sample of size $n$
and recording $\bar{x}$. The distribution of those $\bar{x}$ values is the **sampling
distribution of the mean**. Its standard deviation is called the **standard error**:

$$\operatorname{SE}(\bar{x}) \;=\; \frac{\sigma}{\sqrt{n}} \;\approx\; \frac{s}{\sqrt{n}}$$

Three consequences worth memorising:

1. The standard error is **not** the standard deviation. $s$ describes how spread out the
   *individuals* are; $\operatorname{SE}$ describes how spread out the *estimate* is. Reporting
   one when you mean the other is the single most common mistake in student reports.
2. Precision improves with $\sqrt{n}$, not $n$. To halve the standard error you need **four
   times** the data.
3. The standard deviation $s$ does not shrink as $n$ grows. It converges to $\sigma$. Only the
   standard error shrinks.

When no formula for the standard error exists — the median, the interquartile range, a ratio
of two estimates — the **bootstrap** supplies one. Resample the data with replacement, compute
the statistic on each resample, and take the standard deviation of the results. It replaces
algebra with computation, and it works for any statistic you can write as a function.

---

## 1.4 Confidence intervals

A **95% confidence interval** for the mean is

$$\bar{x} \;\pm\; t_{n-1,\,0.975} \cdot \frac{s}{\sqrt{n}}$$

where $t_{n-1,\,0.975}$ is the 97.5th percentile of the $t$ distribution with $n-1$ degrees of
freedom — `stats.t.ppf(0.975, df=n-1)`. For large $n$ it is close to 1.96; for $n = 5$ it is
2.78, which is why using 1.96 on small samples produces intervals that are too narrow.

**What the 95% refers to.** It is a property of the *procedure*, not of any one interval. If
you repeated the study many times and built an interval each time, about 95% of those intervals
would contain the true $\mu$. The interval you actually have either contains $\mu$ or does not;
the probability statement is about the long run.

Section 2.4 verifies this by simulation rather than asserting it, because the claim is easy to
state and easy to disbelieve.

---

## 1.5 Hypothesis tests

A hypothesis test is a fixed, four-step procedure.

1. **State a null hypothesis $H_0$.** Almost always "there is no effect": the two group means
   are equal, the correlation is zero, the variables are independent.
2. **Compute a test statistic** that measures how far the data are from $H_0$, in units of its
   own standard error. For two means, Welch's $t$:

$$t \;=\; \frac{\bar{x}_1 - \bar{x}_2}{\sqrt{\dfrac{s_1^2}{n_1} + \dfrac{s_2^2}{n_2}}}$$

3. **Find the p-value**: the probability of a test statistic at least this extreme *if $H_0$
   were true*.
4. **Decide**, by comparing the p-value to a threshold $\alpha$ fixed in advance — conventionally
   0.05, a convention with no mathematical justification whatsoever.

**Welch's test, not Student's.** Student's original two-sample test assumes both groups share
one variance. Welch's does not, costs almost nothing when the variances happen to be equal, and
is the better default. In `scipy` that is `equal_var=False`, which is *not* the default — you
must pass it.

---

## 1.6 What a p-value is not

The p-value is $P(\text{data at least this extreme} \mid H_0)$. Almost every misreading comes
from flipping that conditional.

| Statement | Status |
|---|---|
| "p = 0.03, so there is a 3% chance the null is true" | **Wrong.** That is $P(H_0 \mid \text{data})$, a different quantity |
| "p = 0.03, so there is a 97% chance the effect is real" | **Wrong**, same reason |
| "p = 0.20, so the groups are the same" | **Wrong.** Absence of evidence is not evidence of absence |
| "p = 0.001, so the effect is large" | **Wrong.** p measures evidence, not size |
| "p = 0.03, so results this extreme would occur 3% of the time if there were no effect" | Correct |

The fourth row is the important one, and section 2.6 is built around it.

---

## 1.7 Effect size

A p-value confounds two things: how big the effect is, and how much data you have. Write the
two-sample $t$ with equal group sizes and it separates cleanly:

$$t \;=\; d \cdot \sqrt{\frac{n}{2}}, \qquad\text{where}\qquad d \;=\; \frac{\bar{x}_1 - \bar{x}_2}{s_p}$$

$d$ is **Cohen's d**, the difference expressed in pooled standard deviations, with

$$s_p \;=\; \sqrt{\frac{(n_1 - 1)s_1^2 + (n_2 - 1)s_2^2}{n_1 + n_2 - 2}}$$

Because $t$ grows with $\sqrt{n}$ for a fixed $d$, **any** non-zero difference becomes
statistically significant given enough data. Significance therefore tells you that an effect is
detectable. Only the effect size tells you whether it matters.

The usual rough labels are $|d| \approx 0.2$ small, $0.5$ medium, $0.8$ large. They are
conventions, not laws, and a "small" effect on something important can matter more than a
"large" effect on something trivial.

**Report both, always.** A difference of 1.1 marks out of 100 with $p = 0.02$ and a difference
of 12 marks with $p = 0.09$ are both real results, and the second is the more interesting one.

---

## 1.8 Multiple comparisons

If $H_0$ is true, the p-value is uniformly distributed on $[0, 1]$. So a test at $\alpha = 0.05$
produces a false positive 5% of the time, by construction.

Run $k$ independent tests when nothing is real, and the probability that at least one comes out
"significant" is

$$1 - (1 - \alpha)^k$$

For $k = 20$ that is **0.64**. Testing twenty variables against an outcome and reporting the
one that reached $p < 0.05$ is therefore not analysis; it is a demonstration that
$0.95^{20} < 0.5$.

The simplest correction is **Bonferroni**: test each hypothesis at $\alpha / k$. It restores the
family-wise error rate to roughly $\alpha$, at the cost of power. What matters more than the
correction is the habit: decide what you are testing *before* you look, and report how many
tests you ran.

---

## 1.9 Regression: the line and its uncertainty

Simple linear regression fits

$$\hat{y} \;=\; \hat{\beta}_0 + \hat{\beta}_1 x, \qquad \hat{\beta}_1 = r \cdot \frac{s_y}{s_x}$$

by minimising the sum of squared residuals $\sum (y_i - \hat{y}_i)^2$. The slope has its own
standard error,

$$\operatorname{SE}(\hat{\beta}_1) \;=\; \frac{s_e}{s_x \sqrt{n - 1}}$$

where $s_e$ is the residual standard deviation, so the slope gets a confidence interval and a
p-value exactly like a mean does. $R^2$ is the proportion of the variance of $y$ explained by
the fit; for a simple regression it is $r^2$.

Two cautions carried forward from earlier weeks:

- **$R^2$ does not tell you the model is right.** Week 1's Anscombe quartet had four datasets
  with identical fits and identical $R^2$. Section 2.8 fits one of them and looks at the
  residuals, which is where the problem becomes visible.
- **A slope is an association, not a mechanism.** "Each extra hour of study is associated with
  1.6 more marks" is a statement about this data. "Studying one more hour will raise your mark
  by 1.6" is a causal claim that no regression on observational data can support.

---

## 1.10 Categorical association: the chi-square test

For two categorical variables, the null hypothesis is independence. Build the contingency
table of observed counts $O_{ij}$, compute the counts you would expect under independence,

$$E_{ij} \;=\; \frac{(\text{row } i \text{ total}) \times (\text{column } j \text{ total})}{n},$$

and sum the standardised squared differences:

$$\chi^2 \;=\; \sum_{i,j} \frac{(O_{ij} - E_{ij})^2}{E_{ij}}$$

which follows a $\chi^2$ distribution with $(r-1)(c-1)$ degrees of freedom. As with every test,
the statistic grows with $n$, so the effect size — **Cramér's V** — is reported alongside it:

$$V \;=\; \sqrt{\frac{\chi^2}{n \cdot (\min(r, c) - 1)}}$$

A practical warning covered in section 2.9: binning a numeric variable so you can run a
chi-square test throws away most of the information in it, and usually costs you more than the
test gains.

---

---

# Part 2 — Worked example

We build one dataset and use it for the whole session, so that every technique is applied to
the same 480 students rather than to a new toy example each time.

---

## 2.1 The data, and honest description

`build_students` generates an enrolment record: faculty, weekly study hours, attendance, and
an exam mark. The mark is constructed from study hours, attendance, a small faculty offset and
a large amount of noise — so there are real effects in the data, and they are small relative to
individual variation, which is the realistic case.

In [ ]:
FACULTIES = ["Science and Engineering", "Business", "Arts and Humanities", "Education"]

# Faculty differences built into the generator, in marks. These are deliberately small:
# a few marks against an individual spread of about 12 marks.
FACULTY_OFFSET = {
    "Science and Engineering": 2.5,
    "Business": 0.0,
    "Arts and Humanities": -1.5,
    "Education": 1.0,
}


def fix_moments(values, mean, sd):
    """Rescale an array so its sample mean and sample sd are exactly `mean` and `sd`.

    Used only where the lecture quotes an exact figure. It removes sampling noise from
    the generator without removing the shape of the distribution.
    """
    values = np.asarray(values, dtype=float)
    centred = values - values.mean()
    return mean + sd * centred / centred.std(ddof=1)


def build_students(n=480):
    """Generate the enrolment record used throughout this notebook."""
    faculty = rng.choice(FACULTIES, size=n, p=[0.32, 0.30, 0.22, 0.16])
    study_hours = np.clip(rng.gamma(shape=4.0, scale=2.6, size=n), 1.0, 30.0).round(1)
    attendance = np.clip(rng.normal(82.0, 11.0, size=n), 35.0, 100.0).round(1)

    offset = np.array([FACULTY_OFFSET[name] for name in faculty])
    # The true model: 1.6 marks per study hour, 0.18 marks per attendance point,
    # a faculty offset, and residual noise with sd 9.
    exam = (28.0 + 1.6 * study_hours + 0.18 * attendance + offset
            + rng.normal(0.0, 9.0, size=n))
    exam = np.clip(exam, 0.0, 100.0).round(1)

    return pd.DataFrame({
        "student_id": [f"S{i:04d}" for i in range(1, n + 1)],
        "faculty": faculty,
        "study_hours": study_hours,
        "attendance_pct": attendance,
        "exam_score": exam,
        "passed": exam >= 50.0,
    })


students = build_students()
print(f"{len(students)} students, {students['faculty'].nunique()} faculties")
print(students.head())

In [ ]:
def describe_numeric(values) -> pd.Series:
    """Describe one numeric sample, including the standard error of its mean.

    `describe()` gives the spread of the individuals. It does not give the precision
    of the mean, which is what an inferential claim rests on.
    """
    series = pd.Series(values).dropna().astype(float)
    n = series.size
    sd = series.std(ddof=1)
    return pd.Series({
        "n": float(n),
        "mean": series.mean(),
        "sd": sd,
        "se": sd / np.sqrt(n),
        "min": series.min(),
        "q1": series.quantile(0.25),
        "median": series.median(),
        "q3": series.quantile(0.75),
        "max": series.max(),
        "skew": series.skew(),
    })


overall = describe_numeric(students["exam_score"])
print("exam_score, whole cohort")
print(overall.round(3).to_string())

by_faculty = pd.DataFrame({
    name: describe_numeric(group["exam_score"])
    for name, group in students.groupby("faculty")
}).T
print("\nexam_score by faculty")
print(by_faculty[["n", "mean", "sd", "se"]].round(3).to_string())

Read the two columns on the right carefully. `sd` sits between about 10.5 and 13 in every
faculty: individual students vary by roughly 12 marks whichever faculty they are in. `se` is
close to 1: the *average* for a faculty is pinned down to within about a mark.

A three-mark gap between two faculty means is therefore worth discussing. A three-mark gap
between two individual students is noise. The same number means different things depending on
which of the two quantities you divided by.

---

## 2.2 Probability distributions and the `scipy.stats` interface

Every distribution in `scipy.stats` answers the same questions through the same methods. The
one to learn properly is `.ppf`, the inverse CDF, because every critical value in every test
comes from it.

In [ ]:
mu, sigma = 60.0, 12.0

print("Normal(60, 12):")
print(f"  pdf(60)      = {stats.norm.pdf(60, mu, sigma):.5f}   density at the centre")
print(f"  cdf(72)      = {stats.norm.cdf(72, mu, sigma):.4f}    P(X <= 72)")
print(f"  sf(72)       = {stats.norm.sf(72, mu, sigma):.4f}    P(X >  72)  = 1 - cdf")
print(f"  ppf(0.975)   = {stats.norm.ppf(0.975, mu, sigma):.4f}   the 97.5th percentile")
print(f"  ppf(cdf(72)) = {stats.norm.ppf(stats.norm.cdf(72, mu, sigma), mu, sigma):.4f}"
      "    ppf undoes cdf")

print("\nTwo-sided 95% critical value from the t distribution:")
print("     df    t*        vs the normal's 1.9600")
for df in [2, 4, 10, 30, 100, 1000]:
    critical = stats.t.ppf(0.975, df)
    print(f"  {df:>5}   {critical:.4f}    {(critical / 1.959964 - 1) * 100:+.1f}% wider")

In [ ]:
# Three views of the distributions this session uses.
fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.8))

# (a) the normal density, with the upper 2.5% tail shaded
grid = np.linspace(20, 100, 400)
axes[0].plot(grid, stats.norm.pdf(grid, mu, sigma), color=PALETTE[0], lw=2)
cut = stats.norm.ppf(0.975, mu, sigma)
tail = grid[grid >= cut]
axes[0].fill_between(tail, stats.norm.pdf(tail, mu, sigma), color=PALETTE[3], alpha=0.5)
axes[0].axvline(cut, color=PALETTE[3], ls="--", lw=1)
axes[0].annotate(f"ppf(0.975)\n= {cut:.1f}", xy=(cut, 0.006), xytext=(cut + 3, 0.020),
                 arrowprops=dict(arrowstyle="->", color="0.3"), fontsize=9)
axes[0].set_title("(a) Normal(60, 12): the upper 2.5% tail", loc="left")
axes[0].set_xlabel("mark")
axes[0].set_ylabel("density")

# (b) t densities converging on the normal
z = np.linspace(-4.5, 4.5, 400)
axes[1].plot(z, stats.norm.pdf(z), color="black", lw=2, label="normal")
for colour, df in zip(PALETTE[1:], [2, 5, 30]):
    axes[1].plot(z, stats.t.pdf(z, df), color=colour, lw=1.6, label=f"t, df={df}")
axes[1].set_xlim(-4.5, 4.5)
axes[1].set_title("(b) t has heavier tails; df=30 is already close", loc="left")
axes[1].set_xlabel("standardised value")
axes[1].legend(fontsize=8)

# (c) is exam_score normal? A quantile-quantile plot answers it directly.
stats.probplot(students["exam_score"], dist="norm", plot=axes[2])
axes[2].get_lines()[0].set(marker="o", markersize=3, color=PALETTE[0], alpha=0.6)
axes[2].get_lines()[1].set(color=PALETTE[3], lw=1.5)
axes[2].set_title("(c) QQ plot of exam_score against the normal", loc="left")

plt.tight_layout()
plt.show()

shapiro = stats.shapiro(students["exam_score"])
print(f"Shapiro-Wilk on exam_score: W = {shapiro.statistic:.4f}, p = {shapiro.pvalue:.4f}")

Panel (c) is the routine check. If the points follow the line, the sample is consistent with a
normal distribution; systematic curvature means it is not.

A word on the Shapiro–Wilk p-value printed underneath. It is a hypothesis test with
$H_0$ = "the data are normal", and like every test its power grows with $n$. At $n = 480$ it
will reject for departures far too small to matter, and at $n = 15$ it will fail to reject
almost anything. Use the plot to decide; use the test as a footnote.

---

## 2.3 The standard error, two ways

The formula $s/\sqrt{n}$ applies to the mean. The **bootstrap** applies to anything. Both are
computed below for the same sample, and they should agree — which is the point of running them
side by side.

In [ ]:
science = students.loc[students["faculty"] == "Science and Engineering",
                       "exam_score"].to_numpy()

n_science = science.size
sd_science = science.std(ddof=1)
se_formula = sd_science / np.sqrt(n_science)


def bootstrap_distribution(values, statistic=np.mean, n_boot=4000, seed=0):
    """Resample `values` with replacement `n_boot` times and apply `statistic` to each.

    Parameters
    ----------
    values : array-like
        The observed sample.
    statistic : callable
        Applied to each resample. Anything that maps a 1-D array to a number.
    n_boot : int
        Number of resamples.
    seed : int
        Its own generator, so the result does not depend on how many random numbers
        the rest of the notebook has already consumed.

    Returns
    -------
    numpy.ndarray of shape (n_boot,)
    """
    values = np.asarray(values, dtype=float)
    boot_rng = np.random.default_rng(seed)
    index = boot_rng.integers(0, values.size, size=(n_boot, values.size))
    return np.apply_along_axis(statistic, 1, values[index])


boot_means = bootstrap_distribution(science, np.mean, n_boot=4000, seed=12)
boot_medians = bootstrap_distribution(science, np.median, n_boot=4000, seed=12)

print(f"Science and Engineering, n = {n_science}")
print(f"  sd of the students            {sd_science:8.4f}")
print(f"  SE of the mean, s/sqrt(n)     {se_formula:8.4f}")
print(f"  SE of the mean, bootstrap     {boot_means.std(ddof=1):8.4f}")
print(f"  SE of the MEDIAN, bootstrap   {boot_medians.std(ddof=1):8.4f}   (no simple formula)")

In [ ]:
# The standard deviation does not shrink with n. The standard error does.
rows = []
for m in [30, 60, 120, 240, n_science]:
    sub = science[:m]
    rows.append({
        "n": m,
        "sd": sub.std(ddof=1),
        "se": sub.std(ddof=1) / np.sqrt(m),
        "se if n were doubled": sub.std(ddof=1) / np.sqrt(2 * m),
    })
print(pd.DataFrame(rows).set_index("n").round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

axes[0].hist(science, bins=25, color=PALETTE[0], alpha=0.8, edgecolor="white")
axes[0].axvline(science.mean(), color=PALETTE[3], lw=2)
axes[0].set_title(f"(a) the STUDENTS: sd = {sd_science:.2f}", loc="left")
axes[0].set_xlabel("exam score")

axes[1].hist(boot_means, bins=40, color=PALETTE[2], alpha=0.8, edgecolor="white")
axes[1].axvline(science.mean(), color=PALETTE[3], lw=2)
axes[1].set_title(f"(b) the MEAN, over 4000 resamples: SE = {boot_means.std(ddof=1):.2f}",
                  loc="left")
axes[1].set_xlabel("resampled mean exam score")
axes[1].set_xlim(axes[0].get_xlim())          # same x scale — this is the whole point

plt.tight_layout()
plt.show()
print("Both panels share an x-axis. The second is narrow because it describes an estimate,")
print("not a person.")

Plotting the two histograms on a shared x-axis is the clearest way to separate the two
quantities. Panel (a) is how much students differ. Panel (b) is how much the *average* would
move if we ran the study again. They are different questions with different answers, and a
report that shows one while claiming the other is wrong regardless of how the numbers were
computed.

---

## 2.4 Confidence intervals, and what the 95% refers to

In [ ]:
def t_interval(values, level=0.95):
    """Confidence interval for a mean, using the t distribution.

    Returns
    -------
    (low, high) : tuple of float
    """
    values = np.asarray(values, dtype=float)
    n = values.size
    mean = values.mean()
    se = values.std(ddof=1) / np.sqrt(n)
    critical = stats.t.ppf(0.5 + level / 2.0, df=n - 1)
    return mean - critical * se, mean + critical * se


print(f"Science and Engineering mean exam score: {science.mean():.3f}")
for level in [0.80, 0.90, 0.95, 0.99]:
    lo, hi = t_interval(science, level)
    print(f"  {level:.0%} CI  [{lo:.3f}, {hi:.3f}]   width {hi - lo:.3f}")

# The percentile bootstrap gives an interval with no distributional assumption at all.
boot_low, boot_high = np.percentile(boot_means, [2.5, 97.5])
print(f"\n  95% bootstrap interval  [{boot_low:.3f}, {boot_high:.3f}]")
print("  A wider interval is not a worse one. It is a more honest one.")

In [ ]:
def coverage(trials=4000, n=25, true_mean=60.0, true_sd=12.0, level=0.95, use_t=True):
    """Fraction of simulated studies whose interval contains the true mean."""
    samples = rng.normal(true_mean, true_sd, size=(trials, n))
    means = samples.mean(axis=1)
    ses = samples.std(axis=1, ddof=1) / np.sqrt(n)
    critical = (stats.t.ppf(0.5 + level / 2.0, df=n - 1) if use_t
                else stats.norm.ppf(0.5 + level / 2.0))
    return float(np.mean(np.abs(means - true_mean) <= critical * ses))


print("Actual coverage of a nominal 95% interval, 4000 simulated studies each")
print("      n    using t    using 1.96")
for n in [5, 10, 25, 100]:
    print(f"  {n:>5}    {coverage(n=n, use_t=True):.3f}      {coverage(n=n, use_t=False):.3f}")
print("\nThe t column sits on 0.95 at every sample size. The 1.96 column is too low at small n,")
print("which is what 'the interval is too narrow' means in practice.")

In [ ]:
# Forty studies, forty intervals, one true mean.
studies, n_each, true_mean = 40, 25, 60.0
samples = rng.normal(true_mean, 12.0, size=(studies, n_each))

fig, ax = plt.subplots(figsize=(10, 4.2))
misses = 0
for i, sample in enumerate(samples):
    lo, hi = t_interval(sample)
    covers = lo <= true_mean <= hi
    misses += not covers
    colour = PALETTE[0] if covers else PALETTE[3]
    ax.plot([lo, hi], [i, i], color=colour, lw=2, alpha=0.85 if covers else 1.0)
    ax.plot(sample.mean(), i, "o", ms=3, color=colour)

ax.axvline(true_mean, color="black", lw=1.5, ls="--")
ax.set_title(f"40 studies, 40 intervals — {misses} miss the true mean of 60", loc="left")
ax.set_xlabel("exam score")
ax.set_ylabel("study")
ax.set_yticks([])
plt.tight_layout()
plt.show()

print(f"{studies - misses} of {studies} intervals contain the truth.")
print("Each individual interval either does or does not. The 95% is a property of the")
print("procedure across studies, not a probability attached to the one interval you have.")

---

## 2.5 Hypothesis testing, and the p-value's own distribution

First the mechanics, computed by hand and then with `scipy`, so that the function call is not a
black box.

In [ ]:
group_a = rng.normal(60.0, 12.0, size=40)
group_b = rng.normal(65.0, 12.0, size=40)

# --- by hand -------------------------------------------------------------
na, nb = group_a.size, group_b.size
va, vb = group_a.var(ddof=1), group_b.var(ddof=1)
se_difference = np.sqrt(va / na + vb / nb)
t_manual = (group_a.mean() - group_b.mean()) / se_difference

# Welch-Satterthwaite degrees of freedom
df_manual = (va / na + vb / nb) ** 2 / ((va / na) ** 2 / (na - 1) + (vb / nb) ** 2 / (nb - 1))
p_manual = 2 * stats.t.sf(abs(t_manual), df=df_manual)

# --- with scipy ----------------------------------------------------------
result = stats.ttest_ind(group_a, group_b, equal_var=False)
ci = result.confidence_interval(0.95)

print(f"  mean A            {group_a.mean():8.4f}")
print(f"  mean B            {group_b.mean():8.4f}")
print(f"  difference        {group_a.mean() - group_b.mean():8.4f}")
print(f"  SE of difference  {se_difference:8.4f}")
print(f"\n  by hand   t = {t_manual:8.4f}   df = {df_manual:7.3f}   p = {p_manual:.6f}")
print(f"  scipy     t = {result.statistic:8.4f}   df = {result.df:7.3f}   p = {result.pvalue:.6f}")
print(f"\n  95% CI for the difference  [{ci.low:.4f}, {ci.high:.4f}]")
print("\nThe interval is more informative than the p-value: it gives the range of differences")
print("the data are consistent with, in marks.")

In [ ]:
def null_p_values(trials=10000, n=30, mu=60.0, sigma=12.0):
    """p-values from `trials` two-sample tests where both groups come from ONE distribution."""
    a = rng.normal(mu, sigma, size=(trials, n))
    b = rng.normal(mu, sigma, size=(trials, n))
    return stats.ttest_ind(a, b, axis=1, equal_var=False).pvalue


p_null = null_p_values()

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.hist(p_null, bins=20, range=(0, 1), color=PALETTE[0], alpha=0.85, edgecolor="white")
ax.axhline(len(p_null) / 20, color=PALETTE[3], lw=2, ls="--", label="flat (expected)")
ax.axvspan(0, 0.05, color=PALETTE[3], alpha=0.15)
ax.set_title("10,000 experiments in which the null hypothesis is TRUE", loc="left")
ax.set_xlabel("p-value")
ax.set_ylabel("experiments")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

for threshold in [0.01, 0.05, 0.10, 0.50]:
    print(f"  fraction with p < {threshold:<4} :  {np.mean(p_null < threshold):.4f}"
          f"   (expected {threshold:.2f})")

The histogram is flat, and the shaded strip on the left contains about 5% of the experiments.
Nothing is wrong with the test; this is what a p-value *is*. When there is no effect, every
value between 0 and 1 is equally likely, so one experiment in twenty produces a p below 0.05
and looks like a discovery.

Section 2.7 takes that observation and runs twenty tests at once.

---

## 2.6 Significance is not size

Two comparisons of exam marks. Read the group means and the sample sizes before looking at
anything else, and decide which result you would act on.

- **Comparison 1.** A large first-year module taught in two sections, morning and afternoon,
  1200 students in each.
- **Comparison 2.** A pilot of a new tutorial method, 8 students taught the new way and 8
  taught the standard way.

In [ ]:
def build_section_marks(n_per_section=1200):
    """Two large sections of one module. Exact means 62.4 and 63.5, sd 12.0 in both."""
    morning_marks = fix_moments(rng.normal(size=n_per_section), 62.4, 12.0)
    afternoon_marks = fix_moments(rng.normal(size=n_per_section), 63.5, 12.0)
    return pd.DataFrame({
        "section": ["morning"] * n_per_section + ["afternoon"] * n_per_section,
        "mark": np.concatenate([morning_marks, afternoon_marks]).round(1),
    })


sections = build_section_marks()

# A small pilot study. Sixteen marks, recorded individually.
pilot = pd.DataFrame({
    "method": ["new"] * 8 + ["standard"] * 8,
    "mark": [85.0, 64.0, 93.0, 58.0, 80.0, 71.0, 98.0, 61.0,
             70.0, 52.0, 63.0, 81.0, 49.0, 66.0, 58.0, 74.0],
})

for frame, key in [(sections, "section"), (pilot, "method")]:
    print(frame.groupby(key)["mark"].agg(["count", "mean", "std"]).round(3).to_string())
    print()

In [ ]:
def cohens_d(a, b):
    """Standardised difference in means, using the pooled standard deviation."""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    na, nb = a.size, b.size
    pooled = np.sqrt(((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2))
    return (a.mean() - b.mean()) / pooled


def full_comparison(a, b, label):
    """Everything that should be reported about a two-group comparison."""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    outcome = stats.ttest_ind(a, b, equal_var=False)
    interval = outcome.confidence_interval(0.95)
    return {
        "comparison": label,
        "n per group": a.size,
        "difference": a.mean() - b.mean(),
        "CI low": interval.low,
        "CI high": interval.high,
        "t": outcome.statistic,
        "p": outcome.pvalue,
        "cohens_d": cohens_d(a, b),
    }


afternoon = sections.loc[sections["section"] == "afternoon", "mark"]
morning = sections.loc[sections["section"] == "morning", "mark"]
new_method = pilot.loc[pilot["method"] == "new", "mark"]
standard = pilot.loc[pilot["method"] == "standard", "mark"]

table = pd.DataFrame([
    full_comparison(afternoon, morning, "1. afternoon vs morning section"),
    full_comparison(new_method, standard, "2. new vs standard tutorial"),
]).set_index("comparison")

print(table.round(4).to_string())

In [ ]:
row1 = table.iloc[0]
row2 = table.iloc[1]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))

# (a) the large comparison
bins = np.linspace(20, 105, 40)
axes[0].hist(morning, bins=bins, alpha=0.6, label="morning", color=PALETTE[0])
axes[0].hist(afternoon, bins=bins, alpha=0.6, label="afternoon", color=PALETTE[1])
axes[0].axvline(morning.mean(), color=PALETTE[0], lw=2)
axes[0].axvline(afternoon.mean(), color=PALETTE[1], lw=2)
axes[0].set_title(
    "(a) n = 1200 each   difference {:.2f} marks\np = {:.4f}   d = {:.3f}".format(
        row1["difference"], row1["p"], row1["cohens_d"]), loc="left")
axes[0].set_xlabel("mark")
axes[0].legend(fontsize=9)

# (b) the small comparison — every observation drawn, because there are only sixteen
for i, (name, colour) in enumerate([("standard", PALETTE[0]), ("new", PALETTE[1])]):
    values = pilot.loc[pilot["method"] == name, "mark"].to_numpy()
    jitter = np.linspace(-0.07, 0.07, values.size)
    axes[1].plot(values, np.full(values.size, i) + jitter, "o",
                 color=colour, ms=8, alpha=0.85)
    axes[1].plot([values.mean(), values.mean()], [i - 0.22, i + 0.22], color="black", lw=2)

axes[1].set_yticks([0, 1], ["standard", "new"])
axes[1].set_ylim(-0.5, 1.5)
axes[1].set_xlim(40, 105)
axes[1].set_title(
    "(b) n = 8 each   difference {:.2f} marks\np = {:.4f}   d = {:.3f}".format(
        row2["difference"], row2["p"], row2["cohens_d"]), loc="left")
axes[1].set_xlabel("mark")

plt.tight_layout()
plt.show()

In [ ]:
# How large would the pilot have needed to be?
observed_d = abs(table.loc["2. new vs standard tutorial", "cohens_d"])

needed = TTestIndPower().solve_power(effect_size=observed_d, alpha=0.05, power=0.80,
                                     ratio=1.0, alternative="two-sided")

# And the narrower question: at what n would THIS observed effect have reached p < 0.05?
just_significant = next(
    n for n in range(4, 400)
    if observed_d * np.sqrt(n / 2) > stats.t.ppf(0.975, 2 * n - 2)
)

print(f"observed effect size              d = {observed_d:.4f}")
print(f"n per group for 80% power             {needed:.1f}  ->  {int(np.ceil(needed))} students")
print(f"n per group to reach p < 0.05         {just_significant} students (at this exact effect)")
print(f"the pilot actually had                8 students per group")
print("\nThe pilot did not show that the method fails. It was too small to show anything.")

Put the two rows of the table next to each other.

The comparison with $p < 0.05$ has $d \approx 0.09$: the two sections are separated by about a
mark, and a mark is nothing next to a standard deviation of twelve. Twenty-four hundred
students made a trivial difference detectable.

The comparison with $p > 0.05$ has $d \approx 0.92$: the new method's students scored about
twelve marks higher, which would be a substantial result if it held up. Sixteen students were
not enough to establish it.

If you read only the p-values you would report the first and discard the second, which is
exactly backwards. The power calculation makes the second point concrete: a pilot of this size
had roughly a coin-flip chance of detecting an effect this large even if it is entirely real.

**The rule that follows.** Never report a p-value on its own. Report the difference in the
units of the problem, a confidence interval for it, and a standardised effect size. The
p-value is one line of the four, and it is the least informative of them.

---

## 2.7 Twenty tests and one "finding"

Section 2.5 showed that a p-value is uniform when nothing is happening. Here is what that costs
when you test many things at once. The data below contain **no real relationships at all** —
every variable is independent noise.

In [ ]:
def correlation_p_values(predictors, outcome):
    """Pearson r and a two-sided p-value for each ROW of `predictors` against `outcome`.

    Vectorised, because this section runs 20,000 of them. The p-value comes from
    t = r * sqrt((n - 2) / (1 - r**2)), which is t-distributed with n - 2 df under H0.
    """
    x = np.asarray(predictors, dtype=float)
    y = np.asarray(outcome, dtype=float)
    x = x - x.mean(axis=1, keepdims=True)
    y = y - y.mean()
    r = (x @ y) / np.sqrt((x ** 2).sum(axis=1) * (y ** 2).sum())
    n = y.size
    t = r * np.sqrt((n - 2) / (1 - r ** 2))
    return r, 2 * stats.t.sf(np.abs(t), df=n - 2)


# Check the vectorised version against scipy before trusting it.
check_x = rng.normal(size=(3, 50))
check_y = rng.normal(size=50)
r_mine, p_mine = correlation_p_values(check_x, check_y)
r_scipy = stats.pearsonr(check_x[0], check_y)
print(f"row 0   ours   r = {r_mine[0]:.6f}   p = {p_mine[0]:.6f}")
print(f"        scipy  r = {r_scipy.statistic:.6f}   p = {r_scipy.pvalue:.6f}")
assert np.isclose(r_mine[0], r_scipy.statistic) and np.isclose(p_mine[0], r_scipy.pvalue)
print("        they agree.")

In [ ]:
n_students, k_variables = 60, 20

# One "study": 60 students, 20 candidate predictors, one outcome. Nothing is related.
outcome = rng.normal(size=n_students)
predictors = rng.normal(size=(k_variables, n_students))
r_values, p_values = correlation_p_values(predictors, outcome)

trip = pd.DataFrame({
    "variable": [f"v{j:02d}" for j in range(1, k_variables + 1)],
    "r": r_values,
    "p": p_values,
}).sort_values("p").reset_index(drop=True)

print(trip.head(6).round(4).to_string(index=False))
print(f"\nvariables reaching p < 0.05 : {(p_values < 0.05).sum()} of {k_variables}")
print(f"smallest p-value            : {p_values.min():.4f}")
print("\nEvery one of these variables is independent noise.")

In [ ]:
# Repeat the whole study 1000 times and count how often at least one variable "works".
trips = 4000
hits_naive = 0
hits_bonferroni = 0
smallest = np.empty(trips)

for trip_index in range(trips):
    y = rng.normal(size=n_students)
    x = rng.normal(size=(k_variables, n_students))
    _, p = correlation_p_values(x, y)
    smallest[trip_index] = p.min()
    hits_naive += bool((p < 0.05).any())
    hits_bonferroni += bool((p < 0.05 / k_variables).any())

expected_naive = 1 - 0.95 ** k_variables
expected_bonferroni = 1 - (1 - 0.05 / k_variables) ** k_variables

print(f"{trips} studies, {k_variables} independent tests each, nothing real in any of them")
print(f"  at least one p < 0.05        {hits_naive / trips:.3f}"
      f"    theory {expected_naive:.3f}")
print(f"  at least one p < 0.05/20     {hits_bonferroni / trips:.3f}"
      f"    theory {expected_bonferroni:.3f}")

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.hist(smallest, bins=40, range=(0, 1), color=PALETTE[0], alpha=0.85, edgecolor="white")
ax.axvline(0.05, color=PALETTE[3], lw=2, ls="--", label="0.05")
ax.axvline(0.05 / k_variables, color=PALETTE[2], lw=2, ls="--", label="0.05 / 20")
ax.set_title("Smallest of 20 p-values, per study — the null is true every time", loc="left")
ax.set_xlabel("smallest p-value in the study")
ax.set_ylabel("studies")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

Roughly two studies in three produced at least one "significant" variable, and there was
nothing to find in any of them. The distribution of the smallest p-value is nothing like
uniform — taking the minimum of twenty draws pushes it towards zero by construction.

Bonferroni brings the family-wise error rate back to about 5%, at the price of missing real
effects that are not strong. It is a blunt instrument, and it is better than nothing.

What matters more is the reporting habit. If you tested twenty variables, say so. A p-value
reported without the number of tests behind it cannot be interpreted by the reader, because the
same number means different things depending on how many chances it had.

---

## 2.8 Regression: the line, its uncertainty, and its residuals

`statsmodels` is the library for this (McKinney §12.3). The formula interface reads like the
model: `"exam_score ~ study_hours"` is *exam score explained by study hours*.

In [ ]:
model = smf.ols("exam_score ~ study_hours", data=students).fit()
print(model.summary())

In [ ]:
slope = model.params["study_hours"]
slope_ci = model.conf_int().loc["study_hours"]

print(f"intercept          {model.params['Intercept']:.4f}")
print(f"slope              {slope:.4f} marks per study hour")
print(f"  standard error   {model.bse['study_hours']:.4f}")
print(f"  95% CI           [{slope_ci.iloc[0]:.4f}, {slope_ci.iloc[1]:.4f}]")
print(f"  p-value          {model.pvalues['study_hours']:.3e}")
print(f"R-squared          {model.rsquared:.4f}")
print(f"r                  {np.sqrt(model.rsquared):.4f}")
print(f"residual sd        {np.sqrt(model.scale):.4f}")

# The generator used a true slope of 1.6. Did the interval catch it?
print(f"\nThe true slope in build_students is 1.6. Inside the interval: "
      f"{bool(slope_ci.iloc[0] <= 1.6 <= slope_ci.iloc[1])}")

print(f"\nSupported wording:      'each additional study hour is ASSOCIATED WITH about "
      f"{slope:.2f} more marks'")
print("Unsupported wording:    'studying one more hour WILL RAISE your mark'")

In [ ]:
# Two predictors. Each coefficient is that variable's effect holding the other fixed.
model2 = smf.ols("exam_score ~ study_hours + attendance_pct", data=students).fit()
print(model2.summary().tables[1])
print(f"\nR-squared rises from {model.rsquared:.4f} to {model2.rsquared:.4f}")
print(f"attendance coefficient p-value: {model2.pvalues['attendance_pct']:.3e}")

In [ ]:
# Anscombe's second dataset, from Week 1: a respectable R-squared on a model that is wrong.
anscombe2 = pd.DataFrame({
    "x": [10.0, 8.0, 13.0, 9.0, 11.0, 14.0, 6.0, 4.0, 12.0, 7.0, 5.0],
    "y": [9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74],
})
curved = smf.ols("y ~ x", data=anscombe2).fit()

fig, axes = plt.subplots(2, 2, figsize=(12, 7))

# (a) the real data with its fitted line
axes[0, 0].scatter(students["study_hours"], students["exam_score"],
                   s=12, alpha=0.4, color=PALETTE[0])
grid = np.linspace(students["study_hours"].min(), students["study_hours"].max(), 50)
axes[0, 0].plot(grid, model.params["Intercept"] + slope * grid, color=PALETTE[3], lw=2)
axes[0, 0].set_title(f"(a) exam_score ~ study_hours   R-squared = {model.rsquared:.3f}",
                     loc="left")
axes[0, 0].set_xlabel("study hours per week")
axes[0, 0].set_ylabel("exam score")

# (b) its residuals — structureless, which is what you want to see
axes[0, 1].scatter(model.fittedvalues, model.resid, s=12, alpha=0.4, color=PALETTE[0])
axes[0, 1].axhline(0, color=PALETTE[3], lw=1.5)
axes[0, 1].set_title("(b) residuals: no pattern, so a straight line is adequate", loc="left")
axes[0, 1].set_xlabel("fitted value")
axes[0, 1].set_ylabel("residual")

# (c) Anscombe II
axes[1, 0].scatter(anscombe2["x"], anscombe2["y"], s=55, color=PALETTE[1],
                   edgecolor="black", zorder=3)
grid2 = np.linspace(3, 15, 50)
axes[1, 0].plot(grid2, curved.params["Intercept"] + curved.params["x"] * grid2,
                color=PALETTE[3], lw=2)
axes[1, 0].set_title(f"(c) Anscombe II   R-squared = {curved.rsquared:.3f}", loc="left")
axes[1, 0].set_xlabel("x")
axes[1, 0].set_ylabel("y")

# (d) and its residuals, which are an arc
axes[1, 1].scatter(curved.fittedvalues, curved.resid, s=55, color=PALETTE[1],
                   edgecolor="black", zorder=3)
axes[1, 1].axhline(0, color=PALETTE[3], lw=1.5)
axes[1, 1].set_title("(d) residuals: an arc — the relationship is not a line", loc="left")
axes[1, 1].set_xlabel("fitted value")
axes[1, 1].set_ylabel("residual")

plt.tight_layout()
plt.show()

print(f"Anscombe II:  slope {curved.params['x']:.4f}, R-squared {curved.rsquared:.4f}, "
      f"p = {curved.pvalues['x']:.5f}")
print("Every one of those numbers says the model is fine. Panel (d) says it is not.")

Panel (d) is the reason regression output is never read on its own. $R^2 = 0.67$ and
$p = 0.002$ would pass unchallenged in most reports. The residual plot shows the fitted line
running through a curve, missing high in the middle and low at both ends.

**The habit:** fit, then plot residuals against fitted values. A structureless band means the
straight line is adequate. Any visible shape — an arc, a funnel widening to the right, a run of
points on one side — means the model is missing something.

---

## 2.9 Categorical association, and the cost of binning

In [ ]:
def chi_square_report(table, label):
    """Chi-square test of independence, with Cramer's V as the effect size."""
    chi2, p, dof, expected = stats.chi2_contingency(table)
    counts = np.asarray(table)
    n = counts.sum()
    v = np.sqrt(chi2 / (n * (min(counts.shape) - 1)))
    print(label)
    print(table.to_string())
    print(f"  chi-square = {chi2:.4f}   df = {dof}   p = {p:.6f}   Cramer's V = {v:.4f}")
    print(f"  smallest expected count = {expected.min():.2f}"
          "   (the test needs this above about 5)\n")
    return {"chi2": chi2, "p": p, "dof": dof, "v": v}


faculty_table = pd.crosstab(students["faculty"], students["passed"])
faculty_result = chi_square_report(faculty_table, "Faculty against pass/fail")

students["attendance_band"] = pd.cut(
    students["attendance_pct"],
    bins=[0.0, 75.0, 88.0, 101.0],
    labels=["low (<75%)", "medium (75-88%)", "high (>88%)"],
)
attendance_table = pd.crosstab(students["attendance_band"], students["passed"])
attendance_result = chi_square_report(attendance_table, "Attendance band against pass/fail")

In [ ]:
# The same variable, treated two ways.
print("attendance_pct as three BANDS, tested against pass/fail:")
print(f"    chi-square p = {attendance_result['p']:.6f}")
print("\nattendance_pct as a NUMBER, in the regression from section 2.8:")
print(f"    coefficient  = {model2.params['attendance_pct']:.4f} marks per attendance point")
print(f"    p-value      = {model2.pvalues['attendance_pct']:.3e}")
print("\nSame data, same variable. Binning it into three groups and reducing the mark to")
print("pass/fail throws away the exact attendance figure, the exact mark, and the direction")
print("of the effect.")

fig, ax = plt.subplots(figsize=(9, 3.8))
proportions = attendance_table.div(attendance_table.sum(axis=1), axis=0) * 100
bottom = np.zeros(len(proportions))
for column, colour, name in zip(proportions.columns, [PALETTE[3], PALETTE[2]],
                                ["failed", "passed"]):
    ax.bar(proportions.index.astype(str), proportions[column].to_numpy(), bottom=bottom,
           color=colour, label=name, width=0.6)
    bottom = bottom + proportions[column].to_numpy()

for i, band in enumerate(attendance_table.index):
    ax.text(i, 102, f"n = {attendance_table.loc[band].sum()}", ha="center",
            fontsize=9, color="0.3")

ax.set_ylim(0, 112)
ax.set_ylabel("percent of band")
ax.set_title("Pass rate by attendance band", loc="left")
ax.legend(fontsize=9, loc="lower right")
plt.tight_layout()
plt.show()

Two points from this section.

**The chi-square test answers a narrow question.** It says whether the rows and columns are
independent. It does not say which cell drives the result, in which direction, or by how much.
Cramér's V supplies the size; the table itself has to be read for the direction.

**Do not bin a numeric variable in order to test it.** Cutting attendance into three bands and
reducing the mark to pass/fail discards the exact attendance percentage, the exact mark and the
direction of the relationship, and it replaces a well-specified question ("how many marks per
attendance point?") with a vague one ("are these tables different?").

Binning is defensible when the categories are meaningful in the problem — a pass mark, a legal
threshold, a pricing tier. It is not defensible as a way of avoiding a model.

---

---

# Part 3 — Guided workshop (in class)

Four tasks, each building on the last. Every task has a verification cell that recomputes the
answer independently and tells you what is wrong if the numbers do not match.

### Task 1 — Describe, then say how precise

**Problem.** For the **Business** faculty's `exam_score`, assign four variables:

| Variable | Quantity |
|---|---|
| `n_business` | the number of Business students |
| `mean_business` | the mean exam score |
| `sd_business` | the standard deviation, with `ddof=1` |
| `se_business` | the standard error of the mean |

**Hints**

- `students.loc[students["faculty"] == "Business", "exam_score"]` selects the column.
- `Series.std()` already uses `ddof=1`; `ndarray.std()` does not. This distinction was Week 1.
- The standard error is the standard deviation divided by $\sqrt{n}$.

**Expected output.** Four numbers, then `Task 1 complete`. The standard error should be about
a tenth of the standard deviation.

In [ ]:
# TODO 1: describe the Business faculty's exam scores and the precision of their mean.
business = students.loc[students["faculty"] == "Business", "exam_score"]

n_business = None
mean_business = None
sd_business = None
se_business = None

if None not in (n_business, mean_business, sd_business, se_business):
    print(f"n    {n_business}")
    print(f"mean {mean_business:.4f}")
    print(f"sd   {sd_business:.4f}")
    print(f"se   {se_business:.4f}")

In [ ]:
# Verification for Task 1.
reference = students.loc[students["faculty"] == "Business", "exam_score"]
problems = []

if None in (n_business, mean_business, sd_business, se_business):
    problems.append("one or more of the four variables is still None")
else:
    if int(n_business) != reference.size:
        problems.append(f"n_business should be {reference.size}, got {n_business}")
    if not np.isclose(mean_business, reference.mean()):
        problems.append(f"mean_business should be {reference.mean():.4f}, got {mean_business:.4f}")
    if np.isclose(sd_business, reference.std(ddof=0)):
        problems.append("sd_business used ddof=0 — pass ddof=1, or use the pandas Series method")
    elif not np.isclose(sd_business, reference.std(ddof=1)):
        problems.append(f"sd_business should be {reference.std(ddof=1):.4f}, got {sd_business:.4f}")
    expected_se = reference.std(ddof=1) / np.sqrt(reference.size)
    if np.isclose(se_business, reference.std(ddof=1)):
        problems.append("se_business is the standard DEVIATION — divide by sqrt(n)")
    elif not np.isclose(se_business, expected_se):
        problems.append(f"se_business should be {expected_se:.4f}, got {se_business:.4f}")

if problems:
    print("Not complete:")
    for item in problems:
        print("  -", item)
else:
    print(f"Task 1 complete. sd = {sd_business:.3f} describes the students; "
          f"se = {se_business:.3f} describes the mean.")

### Task 2 — A confidence interval, built by hand

**Problem.** Using the four quantities from Task 1, build a **95% confidence interval** for the
Business faculty's mean exam score. Assign `ci_low` and `ci_high`.

Do not call `t_interval`. Build it from the pieces, so that the formula is yours.

**Hints**

- The critical value is `stats.t.ppf(0.975, df=n_business - 1)`.
- The interval is the mean plus and minus the critical value times the standard error.
- Check your critical value: it should be a little above 1.96, not below it.

**Expected output.** Two numbers about two marks apart from the mean on each side, then
`Task 2 complete`.

In [ ]:
# TODO 2: a 95% confidence interval for the Business mean, built from its parts.
critical_value = None
ci_low = None
ci_high = None

if None not in (critical_value, ci_low, ci_high):
    print(f"t*          {critical_value:.4f}")
    print(f"95% CI      [{ci_low:.4f}, {ci_high:.4f}]")
    print(f"half-width  {(ci_high - ci_low) / 2:.4f}")

In [ ]:
# Verification for Task 2.
expected_low, expected_high = t_interval(reference.to_numpy(), 0.95)
expected_critical = stats.t.ppf(0.975, df=reference.size - 1)
problems = []

if None in (critical_value, ci_low, ci_high):
    problems.append("critical_value, ci_low or ci_high is still None")
else:
    if np.isclose(critical_value, 1.959964, atol=1e-4):
        problems.append("critical_value is the NORMAL 1.96 — use stats.t.ppf with df = n - 1")
    elif np.isclose(critical_value, stats.t.ppf(0.95, df=reference.size - 1)):
        problems.append("critical_value used 0.95 — a two-sided 95% interval needs 0.975")
    elif not np.isclose(critical_value, expected_critical, atol=1e-6):
        problems.append(f"critical_value should be {expected_critical:.4f}, got {critical_value:.4f}")

    if ci_low > ci_high:
        problems.append("ci_low is above ci_high — the two are the wrong way round")
    if not np.isclose(ci_low, expected_low, atol=1e-6):
        problems.append(f"ci_low should be {expected_low:.4f}, got {ci_low:.4f}")
    if not np.isclose(ci_high, expected_high, atol=1e-6):
        problems.append(f"ci_high should be {expected_high:.4f}, got {ci_high:.4f}")

if problems:
    print("Not complete:")
    for item in problems:
        print("  -", item)
else:
    print(f"Task 2 complete. The Business mean is {reference.mean():.2f}, and the data are "
          f"consistent with anything from {ci_low:.2f} to {ci_high:.2f}.")

### Task 3 — Test the difference, and report its size

**Problem.** Compare `exam_score` between **Science and Engineering** and **Arts and
Humanities**. Assign three variables:

| Variable | Quantity |
|---|---|
| `t_stat` | Welch's t statistic, Science and Engineering minus Arts and Humanities |
| `p_value` | the two-sided p-value |
| `effect_d` | Cohen's d, same direction |

Then write one sentence in the Markdown cell that follows, reporting the result the way
section 2.6 argued it should be reported.

**Hints**

- `stats.ttest_ind(a, b, equal_var=False)` returns an object with `.statistic` and `.pvalue`.
- The order of the arguments sets the sign. Put Science and Engineering first.
- `cohens_d(a, b)` is defined in section 2.6.

**Expected output.** Three numbers, then `Task 3 complete`.

In [ ]:
# TODO 3: Welch's t-test and Cohen's d for two faculties.
science_scores = students.loc[students["faculty"] == "Science and Engineering", "exam_score"]
arts_scores = students.loc[students["faculty"] == "Arts and Humanities", "exam_score"]

t_stat = None
p_value = None
effect_d = None

if None not in (t_stat, p_value, effect_d):
    print(f"difference  {science_scores.mean() - arts_scores.mean():.4f} marks")
    print(f"t           {t_stat:.4f}")
    print(f"p           {p_value:.6f}")
    print(f"Cohen's d   {effect_d:.4f}")

In [ ]:
# Verification for Task 3.
expected = stats.ttest_ind(science_scores, arts_scores, equal_var=False)
expected_d = cohens_d(science_scores, arts_scores)
problems = []

if None in (t_stat, p_value, effect_d):
    problems.append("t_stat, p_value or effect_d is still None")
else:
    if np.isclose(t_stat, -expected.statistic):
        problems.append("t_stat has the wrong sign — put Science and Engineering first")
    elif not np.isclose(t_stat, expected.statistic, atol=1e-6):
        student_t = stats.ttest_ind(science_scores, arts_scores, equal_var=True)
        if np.isclose(t_stat, student_t.statistic, atol=1e-6):
            problems.append("t_stat used Student's test — pass equal_var=False for Welch's")
        else:
            problems.append(f"t_stat should be {expected.statistic:.4f}, got {t_stat:.4f}")

    if not np.isclose(p_value, expected.pvalue, atol=1e-8):
        if np.isclose(p_value, expected.pvalue / 2, atol=1e-8):
            problems.append("p_value is one-sided — the default two-sided value is wanted")
        else:
            problems.append(f"p_value should be {expected.pvalue:.6f}, got {p_value:.6f}")

    if np.isclose(effect_d, -expected_d):
        problems.append("effect_d has the wrong sign — same order as t_stat")
    elif not np.isclose(effect_d, expected_d, atol=1e-6):
        problems.append(f"effect_d should be {expected_d:.4f}, got {effect_d:.4f}")

if problems:
    print("Not complete:")
    for item in problems:
        print("  -", item)
else:
    verdict = "detectable" if p_value < 0.05 else "not detectable at this sample size"
    print(f"Task 3 complete. The difference is {verdict}, and d = {effect_d:.3f}.")
    print("Both halves of that sentence belong in the report.")

**Write your sentence here.** Report the difference in marks, the confidence interval or the
p-value, and the effect size — in one sentence a non-statistician could read.

*Replace this line with your sentence.*

### Task 4 — Bootstrap an interval for something with no formula

**Problem.** There is no simple standard-error formula for a median. Build a **95% percentile
bootstrap interval** for the median of `students["study_hours"]`, and assign `median_ci_low`
and `median_ci_high`.

**Hints**

- `bootstrap_distribution(values, np.median, n_boot=2000, seed=99)` from section 2.3 returns
  one median per resample.
- A percentile interval is `np.percentile(boot, [2.5, 97.5])`.
- Use at least 2000 resamples. Fewer makes the ends of the interval unstable.

**Expected output.** An interval roughly half a mark-hour wide either side of the sample
median, then `Task 4 complete`.

In [ ]:
# TODO 4: a 95% percentile bootstrap interval for the median study hours.
hours = students["study_hours"].to_numpy()

median_ci_low = None
median_ci_high = None

if None not in (median_ci_low, median_ci_high):
    print(f"sample median  {np.median(hours):.4f}")
    print(f"95% CI         [{median_ci_low:.4f}, {median_ci_high:.4f}]")

In [ ]:
# Verification for Task 4.
# A long, independent bootstrap run gives the target the answer must land near.
target = np.percentile(bootstrap_distribution(hours, np.median, n_boot=20000, seed=2024),
                       [2.5, 97.5])
sample_median = float(np.median(hours))
problems = []

if None in (median_ci_low, median_ci_high):
    problems.append("median_ci_low or median_ci_high is still None")
else:
    if median_ci_low > median_ci_high:
        problems.append("the two ends are the wrong way round")
    if not (median_ci_low <= sample_median <= median_ci_high):
        problems.append(f"the interval must contain the sample median ({sample_median:.3f})")
    if abs(median_ci_low - target[0]) > 0.5:
        problems.append(f"median_ci_low should be near {target[0]:.3f}, got {median_ci_low:.3f}")
    if abs(median_ci_high - target[1]) > 0.5:
        problems.append(f"median_ci_high should be near {target[1]:.3f}, got {median_ci_high:.3f}")
    width = median_ci_high - median_ci_low
    if width > 3.0:
        problems.append(f"the interval is {width:.2f} wide — that is far too wide for n = "
                        f"{hours.size}; check you resampled with replacement")

if problems:
    print("Not complete:")
    for item in problems:
        print("  -", item)
else:
    print(f"Task 4 complete. Median {sample_median:.2f} hours, 95% CI "
          f"[{median_ci_low:.2f}, {median_ci_high:.2f}].")
    print("No formula was used anywhere in that calculation.")

---

# Part 4 — Homework

Two problems. Each has starter code and a block of `assert` statements that checks your work.
Run the assertion cell until it prints `all assertions passed`.

---

## Problem 1 — A complete two-group report

**Problem.** Write `compare_groups(df, value_col, group_col, group_a, group_b)`.

It compares one numeric column between two named groups and returns **everything that should be
reported**, as a dictionary with exactly these keys:

| Key | Value |
|---|---|
| `"n_a"`, `"n_b"` | group sizes, as `int` |
| `"mean_a"`, `"mean_b"` | group means |
| `"difference"` | `mean_a - mean_b` |
| `"t_stat"` | Welch's t statistic |
| `"p_value"` | two-sided p-value |
| `"cohens_d"` | pooled-sd effect size, same direction as `difference` |
| `"ci_low"`, `"ci_high"` | 95% confidence interval **for the difference** |

**Requirements**

- Use Welch's test (`equal_var=False`).
- Raise `KeyError` if `value_col` or `group_col` is not a column.
- Raise `ValueError` if either group label is absent from `group_col`, or has fewer than two
  rows.
- Raise `TypeError` if `value_col` is not numeric.
- Do not modify the input frame.

**Hints**

- `stats.ttest_ind(...).confidence_interval(0.95)` gives `.low` and `.high` for the difference.
- `pd.api.types.is_numeric_dtype(df[value_col])` for the type check.
- `cohens_d` from section 2.6 is already defined; you may call it.
- Check the columns exist **before** you try to select on them, or you will raise the wrong
  error type.

In [ ]:
# Problem 1 — starter code.

def compare_groups(df, value_col: str, group_col: str, group_a, group_b) -> dict:
    """Compare one numeric column between two groups and report the result in full.

    Parameters
    ----------
    df : pandas.DataFrame
        The data. Not modified.
    value_col : str
        Name of the numeric column to compare.
    group_col : str
        Name of the column holding the group labels.
    group_a, group_b : hashable
        The two labels to compare. The difference is group_a minus group_b.

    Returns
    -------
    dict
        Keys: n_a, n_b, mean_a, mean_b, difference, t_stat, p_value,
        cohens_d, ci_low, ci_high.

    Raises
    ------
    KeyError
        If value_col or group_col is not a column of df.
    ValueError
        If either group is absent or has fewer than two rows.
    TypeError
        If value_col is not numeric.
    """
    # TODO: implement as described above.
    raise NotImplementedError("Complete compare_groups()")


demo = compare_groups(students, "exam_score", "faculty",
                      "Science and Engineering", "Arts and Humanities")
for key, value in demo.items():
    print(f"  {key:<12} {value:.4f}" if isinstance(value, float) else f"  {key:<12} {value}")

In [ ]:
# Problem 1 — assertions. Run until this prints "all assertions passed".

# Designed so that every quantity can be worked out on paper.
#   P: 10 12 14 16 18   mean 14, variance 10
#   Q: 20 22 24 26 28   mean 24, variance 10
# difference -10, pooled sd sqrt(10), d = -10/sqrt(10), SE = 2, t = -5, df = 8
sample = pd.DataFrame({
    "group": ["P"] * 5 + ["Q"] * 5 + ["R"],
    "value": [10.0, 12.0, 14.0, 16.0, 18.0, 20.0, 22.0, 24.0, 26.0, 28.0, 99.0],
    "label": ["x"] * 11,
})

out = compare_groups(sample, "value", "group", "P", "Q")

assert isinstance(out, dict), f"Must return a dict; got {type(out).__name__}"
expected_keys = {"n_a", "n_b", "mean_a", "mean_b", "difference",
                 "t_stat", "p_value", "cohens_d", "ci_low", "ci_high"}
assert set(out) == expected_keys, (
    f"Wrong keys.\n  missing: {sorted(expected_keys - set(out))}\n"
    f"  unexpected: {sorted(set(out) - expected_keys)}"
)

assert out["n_a"] == 5 and out["n_b"] == 5, f"Group sizes should be 5 and 5; got {out['n_a']}, {out['n_b']}"
assert isinstance(out["n_a"], (int, np.integer)), "n_a should be an int"
assert np.isclose(out["mean_a"], 14.0), f"mean_a should be 14.0; got {out['mean_a']}"
assert np.isclose(out["mean_b"], 24.0), f"mean_b should be 24.0; got {out['mean_b']}"
assert np.isclose(out["difference"], -10.0), (
    f"difference is mean_a - mean_b = -10.0; got {out['difference']}"
)

assert np.isclose(out["t_stat"], -5.0), (
    f"Welch's t should be exactly -5.0 here; got {out['t_stat']}. "
    "If you got +5.0 the groups are the wrong way round."
)
assert np.isclose(out["p_value"], 2 * stats.t.sf(5.0, df=8)), (
    f"p should be {2 * stats.t.sf(5.0, df=8):.8f} (two-sided, df = 8); got {out['p_value']}"
)
assert np.isclose(out["cohens_d"], -10.0 / np.sqrt(10.0)), (
    f"d should be {-10.0 / np.sqrt(10.0):.6f}; got {out['cohens_d']}"
)

margin = stats.t.ppf(0.975, df=8) * 2.0
assert np.isclose(out["ci_low"], -10.0 - margin), (
    f"ci_low should be {-10.0 - margin:.6f}; got {out['ci_low']}. "
    "The interval is for the DIFFERENCE, not for one group's mean."
)
assert np.isclose(out["ci_high"], -10.0 + margin), (
    f"ci_high should be {-10.0 + margin:.6f}; got {out['ci_high']}"
)
assert out["ci_low"] < out["ci_high"], "ci_low must be below ci_high"

# --- the order of the groups sets the sign ---
flipped = compare_groups(sample, "value", "group", "Q", "P")
assert np.isclose(flipped["difference"], 10.0)
assert np.isclose(flipped["t_stat"], 5.0)
assert np.isclose(flipped["cohens_d"], 10.0 / np.sqrt(10.0))
assert np.isclose(flipped["p_value"], out["p_value"]), "The p-value does not depend on the order"

# --- errors ---
for bad_value, bad_group in [("nope", "group"), ("value", "nope")]:
    try:
        compare_groups(sample, bad_value, bad_group, "P", "Q")
    except KeyError:
        pass
    else:
        raise AssertionError(f"Missing column ({bad_value}, {bad_group}) must raise KeyError")

try:
    compare_groups(sample, "label", "group", "P", "Q")
except TypeError:
    pass
else:
    raise AssertionError("A non-numeric value_col must raise TypeError")

for missing in ["Z", None]:
    try:
        compare_groups(sample, "value", "group", "P", missing)
    except ValueError:
        pass
    else:
        raise AssertionError(f"An absent group label ({missing!r}) must raise ValueError")

try:
    compare_groups(sample, "value", "group", "P", "R")   # R has one row
except ValueError:
    pass
else:
    raise AssertionError("A group with fewer than two rows must raise ValueError")

# --- the input is untouched ---
before = sample.copy()
_ = compare_groups(sample, "value", "group", "P", "Q")
assert sample.equals(before), "compare_groups must not modify its input"

# --- it agrees with the notebook's own analysis ---
real = compare_groups(students, "exam_score", "faculty",
                      "Science and Engineering", "Arts and Humanities")
reference_test = stats.ttest_ind(
    students.loc[students["faculty"] == "Science and Engineering", "exam_score"],
    students.loc[students["faculty"] == "Arts and Humanities", "exam_score"],
    equal_var=False,
)
assert np.isclose(real["t_stat"], reference_test.statistic)
assert np.isclose(real["p_value"], reference_test.pvalue)

print("Problem 1: all assertions passed.")

---

## Problem 2 — A general bootstrap interval

**Problem.** Write `bootstrap_ci(values, statistic=np.mean, n_boot=2000, level=0.95, seed=0)`.

It returns a `(low, high)` tuple: a **percentile bootstrap confidence interval** for whatever
statistic it is given. The point of the function is that it works for statistics that have no
formula — the median, the interquartile range, a trimmed mean, a ratio.

**Requirements**

- Resample `values` **with replacement**, `n_boot` times, each resample the same size as the
  original.
- Apply `statistic` to each resample; return the `(1 - level) / 2` and `(1 + level) / 2`
  percentiles of the results.
- Use **its own** `np.random.default_rng(seed)`. Two calls with the same `seed` must return
  exactly the same interval regardless of what else has drawn random numbers in between.
- Raise `ValueError` if there are fewer than two values, if `n_boot < 100`, or if `level` is
  not strictly between 0 and 1.
- Raise `TypeError` if `statistic` is not callable.
- Do not modify the input.

**Hints**

- `boot_rng.integers(0, n, size=(n_boot, n))` gives all the resample indices at once.
- `np.apply_along_axis(statistic, 1, values[index])` applies the statistic to every row.
- `np.percentile(results, [2.5, 97.5])` for a 95% interval — but compute the two percentages
  from `level` rather than hard-coding them.
- `callable(statistic)` is the type check.

In [ ]:
# Problem 2 — starter code.

def bootstrap_ci(values, statistic=np.mean, n_boot: int = 2000,
                 level: float = 0.95, seed: int = 0):
    """Percentile bootstrap confidence interval for any statistic.

    Parameters
    ----------
    values : array-like
        The observed sample. Not modified.
    statistic : callable
        Applied to each resample; maps a 1-D array to a number.
    n_boot : int
        Number of resamples. At least 100.
    level : float
        Confidence level, strictly between 0 and 1.
    seed : int
        Seed for this function's own generator.

    Returns
    -------
    (low, high) : tuple of float

    Raises
    ------
    ValueError
        If fewer than two values, n_boot < 100, or level is not in (0, 1).
    TypeError
        If statistic is not callable.
    """
    # TODO: implement as described above.
    raise NotImplementedError("Complete bootstrap_ci()")


low, high = bootstrap_ci(students["study_hours"].to_numpy(), np.median, seed=7)
print(f"median study hours   95% CI [{low:.3f}, {high:.3f}]")

low_iqr, high_iqr = bootstrap_ci(
    students["exam_score"].to_numpy(),
    lambda a: np.percentile(a, 75) - np.percentile(a, 25),
    seed=7,
)
print(f"exam score IQR       95% CI [{low_iqr:.3f}, {high_iqr:.3f}]")

In [ ]:
# Problem 2 — assertions. Run until this prints "all assertions passed".

check_rng = np.random.default_rng(1234)
normal_sample = check_rng.normal(50.0, 10.0, size=400)

result = bootstrap_ci(normal_sample, np.mean, n_boot=2000, seed=0)
assert isinstance(result, tuple) and len(result) == 2, (
    f"Must return a (low, high) tuple; got {type(result).__name__}"
)
low, high = result
assert low < high, f"low must be below high; got ({low}, {high})"
assert low <= normal_sample.mean() <= high, (
    "A 95% interval for the mean must contain the sample mean"
)

# --- reproducible, and independent of the global rng ---
first = bootstrap_ci(normal_sample, np.mean, n_boot=2000, seed=0)
_ = rng.normal(size=5000)                      # disturb the notebook's generator
second = bootstrap_ci(normal_sample, np.mean, n_boot=2000, seed=0)
assert first == second, (
    "The same seed must give the same interval. Use your own "
    "np.random.default_rng(seed) inside the function, not the global rng."
)

different = bootstrap_ci(normal_sample, np.mean, n_boot=2000, seed=1)
assert different != first, "A different seed should give a different interval"

# --- it agrees with the t-interval for a mean, where a formula does exist ---
t_low, t_high = t_interval(normal_sample, 0.95)
boot_width = high - low
t_width = t_high - t_low
assert abs(boot_width - t_width) / t_width < 0.20, (
    f"For a mean of 400 normal observations the bootstrap interval should be close to "
    f"the t-interval. Widths: bootstrap {boot_width:.4f}, t {t_width:.4f}."
)

# --- a wider level gives a wider interval ---
narrow = bootstrap_ci(normal_sample, np.mean, n_boot=2000, level=0.80, seed=0)
wide = bootstrap_ci(normal_sample, np.mean, n_boot=2000, level=0.99, seed=0)
assert wide[0] < narrow[0] and narrow[1] < wide[1], (
    "A 99% interval must contain an 80% interval. Compute the percentiles from `level`."
)

# --- statistics with no formula ---
median_ci = bootstrap_ci(normal_sample, np.median, n_boot=2000, seed=3)
assert median_ci[0] <= np.median(normal_sample) <= median_ci[1], (
    "The median interval must contain the sample median"
)

iqr = lambda a: np.percentile(a, 75) - np.percentile(a, 25)
iqr_ci = bootstrap_ci(normal_sample, iqr, n_boot=2000, seed=3)
assert iqr_ci[0] <= iqr(normal_sample) <= iqr_ci[1], (
    "The function must work for any callable, not just np.mean and np.median"
)

# --- resampling really is WITH replacement ---
# Without replacement every resample is a permutation, so every mean is identical
# and the interval collapses to a point.
assert (high - low) > 1e-6, "The interval has zero width — resample WITH replacement"

# --- errors ---
for bad_boot in [0, 50, 99]:
    try:
        bootstrap_ci(normal_sample, np.mean, n_boot=bad_boot)
    except ValueError:
        pass
    else:
        raise AssertionError(f"n_boot={bad_boot} must raise ValueError")

for bad_level in [0.0, 1.0, 1.5, -0.2, 95]:
    try:
        bootstrap_ci(normal_sample, np.mean, level=bad_level)
    except ValueError:
        pass
    else:
        raise AssertionError(f"level={bad_level} must raise ValueError")

for too_few in [[], [1.0]]:
    try:
        bootstrap_ci(too_few, np.mean)
    except ValueError:
        pass
    else:
        raise AssertionError(f"values={too_few} must raise ValueError")

try:
    bootstrap_ci(normal_sample, "mean")
except TypeError:
    pass
else:
    raise AssertionError("A non-callable statistic must raise TypeError")

# --- the input is untouched ---
before = normal_sample.copy()
_ = bootstrap_ci(normal_sample, np.median, n_boot=500, seed=9)
assert np.array_equal(normal_sample, before), "bootstrap_ci must not modify its input"

# --- and it works on a list, not only an array ---
as_list = bootstrap_ci(list(normal_sample), np.mean, n_boot=500, seed=0)
assert isinstance(as_list, tuple) and as_list[0] < as_list[1]

print("Problem 2: all assertions passed.")

---

## Written question

Answer in the Markdown cell below, in 150–250 words.

1. A colleague reports "the new curriculum works: p = 0.03". State precisely what that p-value
   licenses them to say, and name two things they have not told you that you would need before
   agreeing.
2. Section 2.6 compared two studies: one with p = 0.02 and d = 0.09, one with p = 0.09 and
   d = 0.92. Say which of the two you would act on, and what you would do about the other.
3. A second colleague tested twenty variables against exam performance and reports the one that
   reached p = 0.04. Explain what is wrong with that report, and describe two different things
   they could do to make the finding credible.

*Replace this text with your answer.*

**1.**

**2.**

**3.**

---

## Submission checklist

- [ ] All four workshop tasks print their `complete` message.
- [ ] The sentence in Task 3 is written.
- [ ] Problem 1 prints `all assertions passed`.
- [ ] Problem 2 prints `all assertions passed`.
- [ ] The written question is answered in 150–250 words.
- [ ] **Restart Kernel and Run All Cells** completes with no errors.
- [ ] Committed and pushed to your `puc-data-analysis` repository.

## Looking ahead to Week 13

Week 13 is the case study: one real dataset, taken from question to finished analysis using
everything from Weeks 1 to 12. It is also the model for the final project, so the structure of
that session is the structure your project report should follow.

Before the session, read McKinney §13.1–13.5 and bring one question you would like to answer
about a dataset of your own choosing.

---

## References

McKinney, W. (2022). *Python for Data Analysis* (3rd ed.). O'Reilly Media. §12.3.

Pfaffelhuber, P. *Python for Data Analysis*. §9.1–9.5.
Available at `https://pfaffelh.github.io/python_for_data/`.

Anscombe, F. J. (1973). Graphs in statistical analysis. *The American Statistician*, 27(1),
17–21.

Seabold, S., & Perktold, J. (2010). statsmodels: Econometric and statistical modeling with
Python. *Proceedings of the 9th Python in Science Conference*.